# DriftGuard — Streaming Simulation

## About This Notebook

In the previous notebook, I established chronological baseline performance for Gaussian Naive Bayes and KNN.

The Electricity dataset contains a single ordered sequence of observations rather than predefined streaming batches. In this notebook, I will convert that ordered sequence into a simulated stream by dividing it into sequential batches.

The purpose of this simulation is to create a controlled environment in which data arrives over time. This will allow me to evaluate incremental learning, monitor prediction errors, detect concept drift, and adapt the models in the later stages of the project.

### What I Will Do

I will:

- Load the model-ready data
- Preserve the original observation order
- Examine possible batch sizes
- Select and document an appropriate batch size
- Create sequential batches
- Validate that the batching process does not lose or duplicate observations
- Store the streaming configuration and results as project artifacts

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.io import arff

In [2]:
from pathlib import Path
import pandas as pd
from scipy.io import arff
data_path = Path("../data/raw/elecNormNew.arff")
data, metadata = arff.loadarff(data_path)
df = pd.DataFrame(data)
model_data = df.copy()
model_data["day"] = model_data["day"].str.decode("utf-8")
model_data["class"] = model_data["class"].str.decode("utf-8")
model_data = pd.get_dummies(model_data, columns=["day"], dtype=int)
model_data["class"] = model_data["class"].map({"DOWN": 0, "UP": 1})
print("Total observations:", len(model_data))

Total observations: 45312


## Understanding the Stream Size

### Why

Before choosing a batch size, I need to understand how many observations are available and how frequently the models would receive new information under different batch configurations.

The batch size is an experimental design choice. It is not a property supplied by the Electricity dataset.

A very small batch would produce frequent model updates and drift checks, while a very large batch would produce fewer and coarser checkpoints.

I therefore want to compare reasonable candidate sizes before selecting the initial configuration.

In [3]:
# Store the total number of observations in the stream
total_observations = len(model_data)

# Define candidate batch sizes for comparison
candidate_batch_sizes = [100, 250, 500, 1000]

# Calculate the number of batches produced by each candidate size
batch_summary = pd.DataFrame({
    "Batch Size": candidate_batch_sizes,
    "Number of Batches": [
        int(np.ceil(total_observations / size))
        for size in candidate_batch_sizes
    ]
})

# Display the candidate streaming configurations
batch_summary

,Batch Size,Number of Batches
0,100,454
1,250,182
2,500,91
3,1000,46


## Selecting the Initial Batch Size

### Why

I need a batch size that provides enough checkpoints to observe how model performance changes throughout the stream.

A very small batch would cause frequent model updates and produce many evaluation points, while a very large batch would make the simulated system react more slowly to changes.

The Electricity dataset contains 45,312 observations, so I want a batch size that gives enough batches to study performance over time without making the experiment unnecessarily granular.

### Decision

I will use **500 observations per batch** as the initial streaming configuration.

This produces approximately 91 sequential batches across the complete stream. This gives enough checkpoints to observe changes in model behaviour while keeping the streaming experiment computationally practical.

The batch size is an experimental configuration, not a property of the original Electricity dataset. I will keep it consistent across the initial streaming experiments and revisit its sensitivity only if the results indicate that batch granularity materially affects the conclusions.

## Creating the Sequential Stream

### Why

I now need to convert the ordered dataset into the batches that will represent incoming data.

I will create the batches sequentially without shuffling the observations. This preserves the temporal structure of the Electricity stream.

Each observation must appear exactly once in the simulated stream. This is important because accidentally dropping or duplicating observations would change the experiment.

### Decision

Each batch will contain up to 500 consecutive observations. The final batch will contain the remaining observations.

In [4]:
# Define the number of observations that arrive in each batch
batch_size = 500

# Store each sequential batch in a list
batches = []

# Divide the ordered data into consecutive batches
for start in range(0, len(model_data), batch_size):
    end = start + batch_size
    batches.append(model_data.iloc[start:end].copy())

# Display the total number of batches created
print("Number of batches:", len(batches))

# Display the size of the first and final batches
print("First batch size:", len(batches[0]))
print("Final batch size:", len(batches[-1]))

Number of batches: 91
First batch size: 500
Final batch size: 312


## Validating the Streaming Batches

### Why

Creating batches is not enough. I need to verify that the batching process preserved the original dataset correctly.

I will check that:

- The total number of observations is unchanged.
- The first observation remains the first observation.
- The final observation remains the final observation.
- No observations were lost during batching.

This gives me confidence that the streaming simulation is a transformation of the original data rather than a different dataset.

In [5]:
# Count the observations across all created batches
total_batched_observations = sum(len(batch) for batch in batches)

# Confirm that batching preserved the complete dataset
print("Original observations:", len(model_data))
print("Batched observations:", total_batched_observations)

# Check that the first batch starts with the original first observation
print("\nFirst observation preserved:", batches[0].index[0] == model_data.index[0])

# Check that the final batch ends with the original final observation
print("Final observation preserved:", batches[-1].index[-1] == model_data.index[-1])

Original observations: 45312
Batched observations: 45312

First observation preserved: True
Final observation preserved: True


In [6]:
# Create a summary showing the size and index range of each batch
batch_summary = pd.DataFrame({
    "batch_number": range(1, len(batches) + 1),
    "start_index": [batch.index[0] for batch in batches],
    "end_index": [batch.index[-1] for batch in batches],
    "observations": [len(batch) for batch in batches]
})

# Display the first and last few batches
pd.concat([batch_summary.head(), batch_summary.tail()])

,batch_number,start_index,end_index,observations
0,1,0,499,500
1,2,500,999,500
2,3,1000,1499,500
3,4,1500,1999,500
4,5,2000,2499,500
86,87,43000,43499,500
87,88,43500,43999,500
88,89,44000,44499,500
89,90,44500,44999,500
90,91,45000,45311,312


In [7]:
# Store the initial streaming configuration in a small results file
streaming_config = pd.DataFrame({
    "parameter": ["total_observations", "batch_size", "number_of_batches"],
    "value": [len(model_data), batch_size, len(batches)]
})

# Save the streaming configuration for later reference
streaming_config.to_csv("../reports/results/streaming_configuration.csv", index=False)

# Key Findings and Decisions

I converted the ordered Electricity dataset into a simulated sequential stream without changing its original observation order.

### Stream

The complete stream contains 45,312 observations.

### Batch Configuration

I selected a batch size of 500 observations for the initial streaming experiments. This creates 91 sequential batches, with the final batch containing the remaining 312 observations.

### Data Integrity

I verified that the total number of observations remains unchanged after batching and that the first and final observations are preserved.

### Important Methodological Decision

The batches are an experimental representation of data arrival. They are not predefined batches supplied by the Electricity dataset.

I will use the same sequential batching approach when evaluating the incremental and adaptive models unless an experiment explicitly investigates the effect of a different batch size.

### Next Phase

The streaming environment is now established.

In the next notebook, I will implement incremental Gaussian Naive Bayes using Welford's algorithm. Instead of retraining the model from scratch, I will update its learned statistics as each new batch arrives.